# MORGAN fingerprints with Data 1.1

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('SMILES_Big_Data_Set.csv')
df_cdd = df_cdd.rename(columns={
    "GU 10 µM L6: Average Glucose uptake / isoprenaline (%)": "gu",
    "B2 cAMP L6: Average cAMP / isoprenaline (AU)": "camp",
})

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Standard Deviation (±)',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Count',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Standard Deviation (±)',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Count'
                    ], axis=1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display
from rdkit.Chem import Draw

# Sklearn
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# SHAP
import shap

# PyTorch & PyG
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import NNConv, global_mean_pool
from torch_geometric.explain import Explainer, GNNExplainer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('All imports successful.')

In [ ]:
df = df_cdd
df = df.dropna(subset=['camp'])
df.columns = df.columns.str.strip()

# Basic sanity check
required = {'Molecule Name', 'SMILES', 'gu', 'camp'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[['gu','camp']].describe())
df.head()

In [ ]:
# Extract target variables
target_cols = ['gu', 'camp']
colors = ['steelblue', 'darkorange'] 

y = df[target_cols].values
print("Target variables shape:", y.shape)

# Visualize target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes = axes.ravel()

for idx, col in enumerate(target_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color=colors[idx])
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Morgan Fingerprints / ECFP4 (2048-bit)

> Morgan fingerprints (radius=2, nBits=2048) encode circular neighbourhoods around each atom.
> `bitInfo` records which atoms triggered each bit — enabling direct chemical interpretation.

In [ ]:
# ── Morgan fingerprint parameters ──────────────────────────────────────────
MORGAN_RADIUS = 2      # ECFP4
MORGAN_BITS   = 2048

def smiles_to_morgan(smi):
    """Return (fp_array, bitInfo_dict) for a SMILES string."""
    mol = Chem.MolFromSmiles(smi)
    bit_info = {}
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=MORGAN_RADIUS, nBits=MORGAN_BITS, bitInfo=bit_info
    )
    return np.array(fp), bit_info

results      = df['SMILES'].apply(smiles_to_morgan)
X_fp_raw     = np.vstack([r[0] for r in results])   # (N, 2048)
all_bit_info = [r[1] for r in results]               # list of dicts

y = df[['gu', 'camp']].values   # (N, 2)

# Feature names
morgan_names = [f'Morgan_{i}' for i in range(MORGAN_BITS)]

# Remove zero-variance bits
var_mask          = X_fp_raw.var(axis=0) > 0
X                 = X_fp_raw[:, var_mask]
names_filtered    = [morgan_names[i] for i, v in enumerate(var_mask) if v]
# Keep original bit indices for bitInfo lookup
orig_bit_indices  = np.where(var_mask)[0]   # maps filtered col → original bit index

# Aliases used by downstream sections (SVR, RF, XGB, SHAP, consensus)
#X_maccs_filtered  = X_fp               # kept for drop-in compatibility
#maccs_names_filtered = morgan_names_filt

print(f'Morgan FP shape (raw):     {X_fp_raw.shape}')
print(f'Morgan FP shape (filtered): {X.shape}')

# Bit frequency chart
bit_freq = X.mean(axis=0)
plt.figure(figsize=(14, 3))
plt.bar(range(len(bit_freq)), bit_freq, color='steelblue', width=1.0)
plt.xlabel('Morgan bit index (filtered)')
plt.ylabel('Frequency')
plt.title(f'ECFP4 ({MORGAN_BITS}-bit, radius={MORGAN_RADIUS}) Bit Frequencies')
plt.tight_layout()
plt.show()

### 2.1 RDKit bitInfo Interpretation

For each top-scoring Morgan bit, `bitInfo` tells us which atoms (and at what radius) triggered it.
We visualise the atom environment to understand **what chemistry the bit encodes**.

In [ ]:
def get_bit_environment_svg(smiles, bit_idx, bit_info, mol_size=(250, 200)):
    """
    Draw a molecule and highlight ALL atoms belonging to the Morgan environment
    of `bit_idx` using the provided bitInfo dict.
    Returns an SVG string, or None if the bit is not active for this molecule.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or bit_idx not in bit_info:
        return None

    env_atoms = set()
    for center_atom, radius in bit_info[bit_idx]:
        if radius == 0:
            env_atoms.add(center_atom)
        else:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, center_atom)
            amap = set()
            for bond_idx in env:
                bond = mol.GetBondWithIdx(bond_idx)
                amap.add(bond.GetBeginAtomIdx())
                amap.add(bond.GetEndAtomIdx())
            env_atoms.update(amap)

    env_atoms = list(env_atoms)
    drawer = rdMolDraw2D.MolDraw2DSVG(*mol_size)
    rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol, highlightAtoms=env_atoms)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


def show_top_morgan_bits(top_bit_names, smiles_list, all_bit_info, orig_bit_indices,
                          morgan_names_filt, n_mols=3, n_bits=6):
    """
    For each of the top `n_bits` Morgan bits (by name), find up to `n_mols` molecules
    where the bit is active and display the highlighted substructure.
    """
    for bit_name in top_bit_names[:n_bits]:
        filt_idx  = morgan_names_filt.index(bit_name)     # position in filtered array
        orig_bit  = orig_bit_indices[filt_idx]            # original 0-2047 bit index
        print(f'\n── {bit_name}  (original bit {orig_bit}) ──')

        shown = 0
        for smi, binfo in zip(smiles_list, all_bit_info):
            if orig_bit not in binfo:
                continue
            svg = get_bit_environment_svg(smi, orig_bit, binfo)
            if svg:
                print(f'  SMILES: {smi}')
                print(f'  Environments (center_atom, radius): {binfo[orig_bit]}')
                display(SVG(svg))
                shown += 1
                if shown >= n_mols:
                    break
        if shown == 0:
            print('  No active molecules found in dataset.')


# ── Quick demo: show the 5 most frequent bits in the dataset ────────────────
top5_freq_idx  = np.argsort(bit_freq)[::-1][:5]
top5_freq_names = [names_filtered[i] for i in top5_freq_idx]
print('Showing most frequent Morgan bits (structural preview):')
show_top_morgan_bits(
    top5_freq_names, df['SMILES'].tolist(), all_bit_info,
    orig_bit_indices, names_filtered, n_mols=2, n_bits=3
)

### 2.2 SHAP Molecular Highlighting Helper

After SHAP values are computed (§10), this helper maps bit-level SHAP values back onto atoms.
Each atom's colour represents the **sum of SHAP values** of all active Morgan bits that include it.

In [ ]:
import matplotlib.colors as mcolors

def shap_to_atom_weights(smiles, shap_vals_1d, all_bit_info_mol,
                          orig_bit_indices, morgan_names_filt):
    """
    Map per-bit SHAP values onto atoms.
    shap_vals_1d : 1-D array of shape (n_filtered_bits,) — SHAP values for ONE molecule
    Returns: dict {atom_idx: weight}
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    atom_weight = {a.GetIdx(): 0.0 for a in mol.GetAtoms()}

    for filt_idx, shap_val in enumerate(shap_vals_1d):
        orig_bit = orig_bit_indices[filt_idx]
        if orig_bit not in all_bit_info_mol:
            continue
        for center_atom, radius in all_bit_info_mol[orig_bit]:
            if radius == 0:
                atom_weight[center_atom] = atom_weight.get(center_atom, 0) + shap_val
            else:
                env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, center_atom)
                for bond_idx in env:
                    bond = mol.GetBondWithIdx(bond_idx)
                    for a_idx in (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()):
                        atom_weight[a_idx] = atom_weight.get(a_idx, 0) + shap_val
    return atom_weight


def draw_shap_molecule(smiles, shap_vals_1d, all_bit_info_mol,
                        orig_bit_indices, morgan_names_filt,
                        title='', mol_size=(300, 220)):
    """
    Draw a molecule with atoms coloured by their aggregated SHAP contribution.
    Red = positive SHAP (pushes prediction up), Blue = negative.
    """
    atom_weight = shap_to_atom_weights(
        smiles, shap_vals_1d, all_bit_info_mol, orig_bit_indices, morgan_names_filt
    )
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return

    weights    = np.array([atom_weight.get(i, 0) for i in range(mol.GetNumAtoms())])
    max_abs    = max(np.abs(weights).max(), 1e-9)
    norm_w     = weights / max_abs   # [-1, 1]

    cmap = plt.get_cmap('RdBu_r')
    atom_colors = {i: cmap((norm_w[i] + 1) / 2)[:3] for i in range(mol.GetNumAtoms())}
    highlight_atoms = list(range(mol.GetNumAtoms()))

    drawer = rdMolDraw2D.MolDraw2DSVG(*mol_size)
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol,
        highlightAtoms=highlight_atoms,
        highlightAtomColors=atom_colors,
        highlightBonds=[]
    )
    drawer.FinishDrawing()
    if title:
        print(title)
    display(SVG(drawer.GetDrawingText()))


def show_shap_molecules(shap_vals_matrix, target_label, smiles_list,
                         all_bit_info, orig_bit_indices, morgan_names_filt, n=5):
    """
    Display SHAP-highlighted molecule structures for the top-n molecules
    ranked by total absolute SHAP contribution.
    shap_vals_matrix: (N_val, n_filtered_bits)
    """
    total_shap = np.abs(shap_vals_matrix).sum(axis=1)
    top_idx    = np.argsort(total_shap)[::-1][:n]
    print(f'\n=== SHAP Molecular Highlighting — {target_label} (top {n} molecules) ===')
    for rank, mol_idx in enumerate(top_idx):
        draw_shap_molecule(
            smiles_list[mol_idx],
            shap_vals_matrix[mol_idx],
            all_bit_info[mol_idx],
            orig_bit_indices,
            morgan_names_filt,
            title=f'Rank {rank+1}  |  {smiles_list[mol_idx][:60]}'
        )

print('SHAP molecular highlighting helpers defined — call show_shap_molecules() after §10.')

## 3. Cross validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_metrics(y_true, y_pred, target_names=['gu','camp']):
    """Return per-target RMSE, R2."""
    results = {}
    for i, name in enumerate(target_names):
        results[name] = {
            'RMSE': np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            'R2':   r2_score(y_true[:,i], y_pred[:,i])
        }
    return results

def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{'─'*55}')
    print(f'  {model_name} — 5-Fold CV Summary')
    print(f'{'─'*55}')
    for target in ['gu','camp']:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  RMSE: {np.mean(rmse_vals):.4f}±{np.std(rmse_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}±{np.std(r2_vals):.4f}')
    print(f'{'─'*55}')

In [ ]:
# ── Learning Curve Utility ─────────────────────────────────────────────────
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y_col, title, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)):
    """
    Plots train vs. validation RMSE across increasing training set sizes.
    y_col: integer index of the target column to evaluate (one target at a time for
           models that don't natively expose a single score).
    """
    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator,
        X, y[:, y_col],
        cv=cv,
        train_sizes=train_sizes,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        shuffle=True,
        random_state=SEED
    )
    train_rmse = -train_scores
    val_rmse   = -val_scores

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_sizes_abs, train_rmse.mean(axis=1), 'o-', color='steelblue',  label='Train RMSE')
    ax.fill_between(train_sizes_abs,
                    train_rmse.mean(axis=1) - train_rmse.std(axis=1),
                    train_rmse.mean(axis=1) + train_rmse.std(axis=1),
                    alpha=0.15, color='steelblue')
    ax.plot(train_sizes_abs, val_rmse.mean(axis=1),   'o-', color='tomato',     label='Val RMSE')
    ax.fill_between(train_sizes_abs,
                    val_rmse.mean(axis=1) - val_rmse.std(axis=1),
                    val_rmse.mean(axis=1) + val_rmse.std(axis=1),
                    alpha=0.15, color='tomato')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('RMSE')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Print gap at full training size
    gap = val_rmse.mean(axis=1)[-1] - train_rmse.mean(axis=1)[-1]
    print(f"  Train RMSE (full): {train_rmse.mean(axis=1)[-1]:.4f}")
    print(f"  Val   RMSE (full): {val_rmse.mean(axis=1)[-1]:.4f}")
    print(f"  Gap (Val–Train):   {gap:.4f}  {'← possible overfit' if gap > 0.1 else '← ok'}")

## 4. Machine Learning models 
### 4.1 Multi-output support vector regressor

In [ ]:
svr_fold_metrics = []
svr_fold_models  = []   # store (scaler, model) per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)

    # MultiOutputRegressor wraps SVR for 2-target regression
    base_svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
    mo_svr   = MultiOutputRegressor(base_svr, n_jobs=-1)
    mo_svr.fit(X_tr_sc, y_tr)

    y_pred = mo_svr.predict(X_val_sc)
    metrics = cv_metrics(y_val, y_pred)
    svr_fold_metrics.append(metrics)
    svr_fold_models.append((scaler, mo_svr, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(svr_fold_metrics, 'Multi-Output SVR')

### 4.2 Multi-output random forest

In [ ]:
rf_fold_metrics = []
rf_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=SEED, n_jobs=-1
    )
    # RandomForestRegressor natively handles multi-output
    rf.fit(X_tr, y_tr)

    y_pred = rf.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    rf_fold_metrics.append(metrics)
    rf_fold_models.append((rf, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(rf_fold_metrics, 'Multi-Output Random Forest')

### 4.3 Multi-output XGBoost

In [ ]:
xgb_fold_metrics = []
xgb_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # XGBoost ≥ 2.0 supports multi-output natively with multi_strategy
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        multi_strategy='multi_output_tree',   # native multi-output
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = xgb_model.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    xgb_fold_metrics.append(metrics)
    xgb_fold_models.append((xgb_model, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(xgb_fold_metrics, 'Multi-Output XGBoost')

### 4.4 Model performance comparison

In [ ]:
def aggregate_cv(fold_metrics, model_name):
    rows = []
    for target in ['gu','camp']:
        for metric in ['RMSE','R2']:
            vals = [m[target][metric] for m in fold_metrics]
            rows.append({'Model': model_name, 'Target': target,
                         'Metric': metric, 'Mean': np.mean(vals), 'Std': np.std(vals)})
    return pd.DataFrame(rows)

results_df = pd.concat([
    aggregate_cv(svr_fold_metrics,  'SVR'),
    aggregate_cv(rf_fold_metrics,   'Random Forest'),
    aggregate_cv(xgb_fold_metrics,  'XGBoost'),
], ignore_index=True)

# Pivot for readability
pivot = results_df.pivot_table(index=['Model','Target'], columns='Metric', values=['Mean','Std'])
print(pivot.round(4))

# Bar chart with best model highlighted in green
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['gu', 'camp']):
    sub = results_df[(results_df['Target'] == target) & (results_df['Metric'] == 'RMSE')]

    # Identify best model (lowest RMSE)
    best_idx = sub['Mean'].idxmin()
    best_model = sub.loc[best_idx, 'Model']

    # Assign colors: green for best, blue for others
    colors = ['green' if m == best_model else 'steelblue' for m in sub['Model']]

    ax.bar(sub['Model'], sub['Mean'], yerr=sub['Std'], 
           capsize=5, color=colors, alpha=0.85)

    ax.set_title(f'{target.upper()} — Mean RMSE (5-Fold CV)')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


print('\nExpected output: bar chart + pivot table with RMSE/MAE/R2 for all 4 models on both targets.')

### 4.5. Overfitting / underfitting diagnostics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OVERFITTING / UNDERFITTING DIAGNOSTICS
# Run AFTER Cells 14 (SVR), 16 (RF), and 18 (XGB) have been executed.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import r2_score, mean_squared_error

TARGETS = ['gu', 'camp']
TARGET_LABELS = ['GU', 'cAMP']

# ─────────────────────────────────────────────────────────────────────────────
# 1. TRAIN vs. VALIDATION RMSE  (bias-variance gap)
# ─────────────────────────────────────────────────────────────────────────────
# Re-evaluate each stored fold model on its own TRAINING split

def compute_train_val_rmse(fold_models, model_type, X, y):
    """
    Returns two dicts keyed by target: lists of (train_rmse, val_rmse) per fold.
    model_type: 'svr' | 'rf' | 'xgb'
    """
    train_rmses = {t: [] for t in TARGETS}
    val_rmses   = {t: [] for t in TARGETS}

    for entry in fold_models:
        if model_type == 'svr':
            scaler, model, tr_idx, val_idx = entry
            X_tr  = scaler.transform(X[tr_idx])
            X_val = scaler.transform(X[val_idx])
        else:
            model, tr_idx, val_idx = entry
            X_tr  = X[tr_idx]
            X_val = X[val_idx]

        y_tr_pred  = model.predict(X_tr)
        y_val_pred = model.predict(X_val)

        for i, t in enumerate(TARGETS):
            train_rmses[t].append(np.sqrt(mean_squared_error(y[tr_idx, i],  y_tr_pred[:, i])))
            val_rmses[t].append(  np.sqrt(mean_squared_error(y[val_idx, i], y_val_pred[:, i])))

    return train_rmses, val_rmses


svr_tr_rmse, svr_val_rmse  = compute_train_val_rmse(svr_fold_models,  'svr', X, y)
rf_tr_rmse,  rf_val_rmse   = compute_train_val_rmse(rf_fold_models,   'rf',  X, y)
xgb_tr_rmse, xgb_val_rmse  = compute_train_val_rmse(xgb_fold_models,  'xgb', X, y)

model_results = {
    'SVR':           (svr_tr_rmse,  svr_val_rmse),
    'Random Forest': (rf_tr_rmse,   rf_val_rmse),
    'XGBoost':       (xgb_tr_rmse,  xgb_val_rmse),
}

folds = np.arange(1, 6)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharey='row')
fig.suptitle('Train vs. Validation RMSE per Fold\n'
             '(Large gap = overfitting  |  Both high = underfitting)',
             fontsize=13, fontweight='bold')

for col, (mname, (tr_rmse, val_rmse)) in enumerate(model_results.items()):
    for row, t in enumerate(TARGETS):
        ax = axes[row, col]
        ax.plot(folds, tr_rmse[t],  'o-', color='steelblue',  label='Train RMSE', lw=2)
        ax.plot(folds, val_rmse[t], 's--', color='tomato',    label='Val RMSE',   lw=2)
        ax.fill_between(folds, tr_rmse[t], val_rmse[t], alpha=0.15, color='orange',
                        label='Gap (overfit zone)')
        ax.set_title(f'{mname}  [{TARGET_LABELS[row]}]')
        ax.set_xlabel('Fold')
        ax.set_ylabel('RMSE')
        ax.legend(fontsize=8)
        ax.set_xticks(folds)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 2. SUMMARY TABLE  — mean train RMSE vs. mean val RMSE + gap ratio
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for mname, (tr_rmse, val_rmse) in model_results.items():
    for t in TARGETS:
        tr_mean  = np.mean(tr_rmse[t])
        val_mean = np.mean(val_rmse[t])
        gap      = val_mean - tr_mean
        ratio    = val_mean / tr_mean if tr_mean > 0 else np.nan

        if ratio > 2.0:
            diagnosis = '⚠️  Overfitting'
        elif val_mean > 0.5 * y[:, TARGETS.index(t)].std() * 2:
            diagnosis = '⚠️  Underfitting'
        else:
            diagnosis = '✅  Good fit'

        rows.append({
            'Model':          mname,
            'Target':         t,
            'Train RMSE':     round(tr_mean,  4),
            'Val RMSE':       round(val_mean, 4),
            'Gap (Val-Train)':round(gap,       4),
            'Val/Train ratio':round(ratio,     2),
            'Diagnosis':      diagnosis,
        })

summary_df = pd.DataFrame(rows)
print('\n── Overfitting / Underfitting Summary ──────────────────────────────────')
print(summary_df.to_string(index=False))


# ─────────────────────────────────────────────────────────────────────────────
# 3. PREDICTED vs. ACTUAL  (best fold of each model)
# ─────────────────────────────────────────────────────────────────────────────
def get_best_val_preds(fold_models, model_type, fold_metrics, X, y):
    best = np.argmin([(m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in fold_metrics])
    entry = fold_models[best]
    if model_type == 'svr':
        scaler, model, tr_idx, val_idx = entry
        X_val = scaler.transform(X[val_idx])
    else:
        model, tr_idx, val_idx = entry
        X_val = X[val_idx]
    return y[val_idx], model.predict(X_val)

svr_yv, svr_yp  = get_best_val_preds(svr_fold_models,  'svr', svr_fold_metrics,  X, y)
rf_yv,  rf_yp   = get_best_val_preds(rf_fold_models,   'rf',  rf_fold_metrics,   X, y)
xgb_yv, xgb_yp  = get_best_val_preds(xgb_fold_models,  'xgb', xgb_fold_metrics,  X, y)

model_preds = [
    ('SVR',           svr_yv, svr_yp),
    ('Random Forest', rf_yv,  rf_yp),
    ('XGBoost',       xgb_yv, xgb_yp),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Predicted vs. Actual  (Best Fold)\n'
             'Points on the diagonal = perfect prediction',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tname) in enumerate(zip(range(2), TARGET_LABELS)):
        ax = axes[row, col]
        ax.scatter(yv[:, ti], yp[:, ti], alpha=0.6, edgecolors='k', linewidths=0.3,
                   color='steelblue', s=40)
        lo = min(yv[:, ti].min(), yp[:, ti].min())
        hi = max(yv[:, ti].max(), yp[:, ti].max())
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Ideal')
        r2 = r2_score(yv[:, ti], yp[:, ti])
        ax.set_title(f'{mname}  [{tname}]  R²={r2:.3f}')
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 4. RESIDUAL PLOTS  (best fold)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Residuals vs. Predicted  (Best Fold)\n'
             'Patterns = model misfit  |  Funnel shape = heteroscedasticity',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tname) in enumerate(zip(range(2), TARGET_LABELS)):
        ax = axes[row, col]
        residuals = yv[:, ti] - yp[:, ti]
        ax.scatter(yp[:, ti], residuals, alpha=0.6, edgecolors='k',
                   linewidths=0.3, color='darkorange', s=40)
        ax.axhline(0, color='red', linestyle='--', lw=1.5)
        ax.set_title(f'{mname}  [{tname}]')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Residual (Actual − Predicted)')

plt.tight_layout()
plt.show()

print('\nDiagnostic complete.')
print('Interpretation guide:')
print('  Train/Val RMSE gap  >  2×  →  likely overfitting')
print('  Both train & val RMSE high  →  likely underfitting')
print('  Predicted vs Actual: points hugging the diagonal  →  good fit')
print('  Residuals: no pattern around zero  →  unbiased model')

## 5. Helper — Bias Score for High GU + Low cAMP

The **bias score** for a feature $f$ across samples is computed on **normalised** SHAP values:

$$\\hat{\\phi}_{gu}(f) = \\frac{\\overline{\\phi}_{gu}(f)}{\\max_j |\\overline{\\phi}_{gu}(j)|}, \\quad \\hat{\\phi}_{camp}(f) = \\frac{\\overline{\\phi}_{camp}(f)}{\\max_j |\\overline{\\phi}_{camp}(j)|}$$

$$\\text{BiasScore}(f) = \\hat{\\phi}_{gu}(f) - \\hat{\\phi}_{camp}(f)$$

**Why normalise?** GU and cAMP can have very different SHAP magnitude scales. Without normalisation the target with larger absolute SHAP values dominates the subtraction, obscuring the true feature preference. Dividing each target's mean SHAP vector by its own max-absolute value maps both onto $[-1, 1]$ before subtraction, so each target contributes equally to the score.

A positive bias score means the feature pushes toward **high GU AND low cAMP** — the desired combination.

In [ ]:
def bias_score_plot(shap_gu, shap_camp, feature_names, top_n=20, title='Bias Score'):
    """
    shap_gu, shap_camp : (N, F) SHAP value arrays

    Normalised bias score:
        mean_gu   is divided by max(|mean_gu|)   → range [-1, 1]
        mean_camp is divided by max(|mean_camp|) → range [-1, 1]
        bias = norm_gu - norm_camp

    This ensures both targets contribute equally regardless of their
    absolute SHAP magnitude scales.
    Positive → feature promotes high GU and/or low cAMP
    """
    mean_gu   = shap_gu.mean(axis=0)
    mean_camp = shap_camp.mean(axis=0)

    # ── NEW: per-target normalisation to [-1, 1] ──────────────────────────
    max_gu   = np.abs(mean_gu).max()
    max_camp = np.abs(mean_camp).max()
    norm_gu   = mean_gu   / max_gu   if max_gu   > 0 else mean_gu
    norm_camp = mean_camp / max_camp if max_camp > 0 else mean_camp
    bias = norm_gu - norm_camp  # range approx [-2, 2]
    # ──────────────────────────────────────────────────────────────────────

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]
    top_names  = [feature_names[i] for i in idx_sorted]
    top_bias   = bias[idx_sorted]

    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]
    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Normalised Bias Score  (↑ favours High GU & Low cAMP)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    top_df = pd.DataFrame({
        'Feature':        top_names,
        'BiasScore':      top_bias,
        'NormSHAP_gu':    norm_gu[idx_sorted],
        'NormSHAP_camp':  norm_camp[idx_sorted],
        'RawMeanSHAP_gu':   mean_gu[idx_sorted],
        'RawMeanSHAP_camp': mean_camp[idx_sorted],
    })
    return top_df

print('Normalised bias-score helper defined.')


## 6. Explainable AI methods
### 6.1 KernelSHAP — SVR

KernelSHAP is model-agnostic and the appropriate SHAP method for SVR.

In [ ]:
# Use best fold (lowest mean RMSE across targets)
best_fold_svr = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in svr_fold_metrics
])
scaler_svr, mo_svr_best, tr_idx, val_idx = svr_fold_models[best_fold_svr]

X_tr_sc_svr  = scaler_svr.transform(X[tr_idx])
X_val_sc_svr = scaler_svr.transform(X[val_idx])

# KernelSHAP uses a background summary (k-means to keep it tractable)
background = shap.kmeans(X_tr_sc_svr, 30)

shap_svr_gu, shap_svr_camp = [], []

for i, estimator in enumerate(mo_svr_best.estimators_):
    print(f'  Computing KernelSHAP for SVR output {i+1}/2 ...')
    explainer = shap.KernelExplainer(estimator.predict, background)
    sv = explainer.shap_values(X_val_sc_svr, nsamples=200)
    if i == 0:
        shap_svr_gu = sv
    else:
        shap_svr_camp = sv

# ── Individual SHAP summary plots ─────────────────────────────────────────
print('\n[SVR] SHAP Summary — GU:')
shap.summary_plot(shap_svr_gu, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

print('\n[SVR] SHAP Summary — cAMP:')
shap.summary_plot(shap_svr_camp, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

# ── Bias score ─────────────────────────────────────────────────────────────
print('\n[SVR] Bias Score — High GU & Low cAMP:')
svr_bias_df = bias_score_plot(
    np.array(shap_svr_gu), np.array(shap_svr_camp),
    names_filtered, top_n=20, title='SVR — Bias Score (High GU & Low cAMP)'
)
display(svr_bias_df.head(10))

### 6.2 TreeSHAP — Random Forest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display

# ─────────────────────────────────────────────────────────────
# Select best fold
# ─────────────────────────────────────────────────────────────
best_fold_rf = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2
    for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]
X_tr_rf  = X[tr_idx_rf]
X_val_rf = X[val_idx_rf]

names_filtered = np.asarray(names_filtered).ravel().tolist()

# ─────────────────────────────────────────────────────────────
# Detect model type once
# ─────────────────────────────────────────────────────────────
is_multioutput_wrapper = (
    hasattr(rf_best, "estimators_") and
    not hasattr(rf_best, "feature_importances_")
)

# ─────────────────────────────────────────────────────────────
# (a) Feature Importance (MDI)
# ─────────────────────────────────────────────────────────────
if is_multioutput_wrapper:
    fi_gu   = rf_best.estimators_[0].feature_importances_
    fi_camp = rf_best.estimators_[1].feature_importances_
    fi_per_output = np.vstack([fi_gu, fi_camp])
else:
    fi = rf_best.feature_importances_
    fi_per_output = np.vstack([fi, fi])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, target_i, target_name in zip(axes, [0, 1], ['GU', 'cAMP']):
    top20_idx = np.argsort(fi_per_output[target_i])[::-1][:20]
    ax.barh(
        [names_filtered[i] for i in top20_idx][::-1],
        fi_per_output[target_i][top20_idx][::-1]
    )
    ax.set_title(f'RF MDI Importance — {target_name}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────
# (b) TreeSHAP for Multi-Output RF
# ─────────────────────────────────────────────────────────────
def extract_shap_2d(shap_vals, output_idx=0):
    """
    Normalize SHAP output to a 2D array (n_samples, n_features).
    Handles:
      - list of arrays  → pick output_idx
      - 3D ndarray      → (n_samples, n_features, n_outputs), slice output_idx
      - 2D ndarray      → return as-is
    """
    if isinstance(shap_vals, list):
        arr = np.array(shap_vals[output_idx])
    elif isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        arr = shap_vals[:, :, output_idx]
    else:
        arr = np.array(shap_vals)
    if arr.ndim != 2:
        raise ValueError(f"Unexpected SHAP array shape after extraction: {arr.shape}")
    return arr

if is_multioutput_wrapper:
    # Each estimator is a single-output RF
    rf_explainer_gu   = shap.TreeExplainer(rf_best.estimators_[0], X_tr_rf,
                                            feature_perturbation='interventional')
    rf_explainer_camp = shap.TreeExplainer(rf_best.estimators_[1], X_tr_rf,
                                            feature_perturbation='interventional')
    shap_rf_gu_raw   = rf_explainer_gu.shap_values(X_val_rf,   check_additivity=False)
    shap_rf_camp_raw = rf_explainer_camp.shap_values(X_val_rf, check_additivity=False)
    shap_rf_gu   = extract_shap_2d(shap_rf_gu_raw,   output_idx=0)
    shap_rf_camp = extract_shap_2d(shap_rf_camp_raw, output_idx=0)
else:
    # Native multi-output RF — use interventional perturbation + disable additivity check
    rf_explainer = shap.TreeExplainer(rf_best, X_tr_rf,
                                       feature_perturbation='interventional')
    shap_rf_raw  = rf_explainer.shap_values(X_val_rf, check_additivity=False)
    shap_rf_gu   = extract_shap_2d(shap_rf_raw, output_idx=0)
    shap_rf_camp = extract_shap_2d(shap_rf_raw, output_idx=1)

print('\n[RF] SHAP Beeswarm — GU:')
shap.summary_plot(shap_rf_gu,   X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] SHAP Beeswarm — cAMP:')
shap.summary_plot(shap_rf_camp, X_val_rf, feature_names=names_filtered, max_display=20)

# ─────────────────────────────────────────────────────────────
# (c) Bias Score
# ─────────────────────────────────────────────────────────────
def bias_score_plot(shap_gu, shap_camp, feature_names, top_n=20, title='Bias Score'):
    feature_names = np.asarray(feature_names).ravel().tolist()
    shap_gu   = np.array(shap_gu)
    shap_camp = np.array(shap_camp)
    mean_gu   = np.abs(shap_gu).mean(axis=0)
    mean_camp = np.abs(shap_camp).mean(axis=0)
    bias = mean_gu - mean_camp

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]
    top_names  = [feature_names[int(i)] for i in idx_sorted]
    top_bias   = bias[idx_sorted]
    colors     = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]

    plt.figure(figsize=(8, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=1)
    plt.title(title, fontweight='bold')
    plt.xlabel('Bias (|SHAP_GU| − |SHAP_cAMP|)')
    plt.tight_layout()
    plt.show()

    return pd.DataFrame({
        "Feature":          top_names,
        "Bias Score":       top_bias,
        "GU Mean |SHAP|":   mean_gu[idx_sorted],
        "cAMP Mean |SHAP|": mean_camp[idx_sorted]
    })

# ─────────────────────────────────────────────────────────────
# (d) Compute Bias
# ─────────────────────────────────────────────────────────────
print('\n[RF] Bias Score — High GU & Low cAMP:')
rf_bias_df = bias_score_plot(
    shap_rf_gu, shap_rf_camp,
    names_filtered, top_n=20,
    title='Random Forest — Bias Score (High GU & Low cAMP)'
)
display(rf_bias_df.head(10))

### 6.3 TreeSHAP + Feature Importance — XGBoost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# ─────────────────────────────────────────────────────────────────────────────
# Select best fold
# ─────────────────────────────────────────────────────────────────────────────
best_fold_xgb = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2
    for m in xgb_fold_metrics
])
xgb_best, tr_idx_xgb, val_idx_xgb = xgb_fold_models[best_fold_xgb]
X_tr_xgb  = X[tr_idx_xgb]
X_val_xgb = X[val_idx_xgb]

# ─────────────────────────────────────────────────────────────────────────────
# (a) SHAP-Based Feature Importance (KernelExplainer for multi-output XGBoost)
# TreeExplainer does not support vector-leaf (multi-output) XGBoost models,
# so we use KernelExplainer with per-output prediction wrappers instead.
# ─────────────────────────────────────────────────────────────────────────────

def predict_gu(x):
    return xgb_best.predict(x)[:, 0]

def predict_camp(x):
    return xgb_best.predict(x)[:, 1]

bg = shap.kmeans(X_tr_xgb, 30)

print("Computing SHAP values for GU (this may take a moment)...")
ex_gu = shap.KernelExplainer(predict_gu, bg)
shap_xgb_gu = ex_gu.shap_values(X_val_xgb, nsamples=200)

print("Computing SHAP values for cAMP...")
ex_camp = shap.KernelExplainer(predict_camp, bg)
shap_xgb_camp = ex_camp.shap_values(X_val_xgb, nsamples=200)

# Compute mean absolute SHAP importance across both targets
fi_xgb = (
    np.abs(shap_xgb_gu).mean(axis=0) +
    np.abs(shap_xgb_camp).mean(axis=0)
) / 2
top20_xgb = np.argsort(fi_xgb)[::-1][:20]

plt.figure(figsize=(10, 6))
plt.barh(
    [names_filtered[i] for i in top20_xgb][::-1],
    fi_xgb[top20_xgb][::-1]
)
plt.xlabel("Mean |SHAP value|")
plt.title("XGBoost Feature Importance (Multi-Output, SHAP-Based)")
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# (b) SHAP Beeswarm Plots
# ─────────────────────────────────────────────────────────────────────────────
print('\n[XGB] SHAP Beeswarm — GU:')
shap.summary_plot(
    shap_xgb_gu,
    X_val_xgb,
    feature_names=names_filtered,
    max_display=20
)

print('\n[XGB] SHAP Beeswarm — cAMP:')
shap.summary_plot(
    shap_xgb_camp,
    X_val_xgb,
    feature_names=names_filtered,
    max_display=20
)

# ─────────────────────────────────────────────────────────────────────────────
# (c) Bias Score Analysis
# ─────────────────────────────────────────────────────────────────────────────
print('\n[XGB] Bias Score — High GU & Low cAMP:')
xgb_bias_df = bias_score_plot(
    shap_xgb_gu,
    shap_xgb_camp,
    names_filtered,
    top_n=20,
    title='XGBoost — Bias Score (High GU & Low cAMP)'
)
display(xgb_bias_df.head(10))

## 7. Consensus Bias Score Across Models

Aggregate MACCS-level bias scores from SVR, RF, and XGBoost to find features that **all models agree** drive high GU + low cAMP.

In [ ]:
def get_bias_vector(shap_gu, shap_camp):
    """Mean SHAP GU - mean SHAP cAMP per feature."""
    return shap_gu.mean(axis=0) - shap_camp.mean(axis=0)

bias_svr  = get_bias_vector(np.array(shap_svr_gu),   np.array(shap_svr_camp))
bias_rf   = get_bias_vector(shap_rf_gu,   shap_rf_camp)
bias_xgb  = get_bias_vector(shap_xgb_gu,  shap_xgb_camp)

# Normalise each to [-1, 1] before averaging
def normalise(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

consensus_bias = (normalise(bias_svr) + normalise(bias_rf) + normalise(bias_xgb)) / 3.0

top_n = 25
idx_top = np.argsort(np.abs(consensus_bias))[::-1][:top_n]
names_top = [names_filtered[i] for i in idx_top]
vals_top  = consensus_bias[idx_top]

colors = ['#1976D2' if v > 0 else '#D32F2F' for v in vals_top]
plt.figure(figsize=(11, 7))
plt.barh(names_top[::-1], vals_top[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Normalised Consensus Bias Score')
plt.title('Consensus Bias Score (SVR + RF + XGBoost)\n↑ = High Glucose Uptake & Low cAMP', fontsize=13)
plt.tight_layout()
plt.show()

consensus_df = pd.DataFrame({
    'Feature':        names_top,
    'ConsensusScore': vals_top,
    'SVR_bias':  bias_svr[idx_top],
    'RF_bias':   bias_rf[idx_top],
    'XGB_bias':  bias_xgb[idx_top]
}).sort_values('ConsensusScore', ascending=False)

print('Top features driving HIGH GU & LOW cAMP (positive score):')
display(consensus_df[consensus_df['ConsensusScore'] > 0].head(10))
print('\nTop features that hinder the desired outcome (negative score):')
display(consensus_df[consensus_df['ConsensusScore'] < 0].head(10))

In [ ]:
# ── Molecules containing each top consensus bit ───────────────────────────────
def molecules_with_bit(bit_name, morgan_names_filt, orig_bit_indices,
                        all_bit_info, df):
    """
    Return a DataFrame of all molecules in which `bit_name` is active,
    identified by Molecule Name, SMILES, gu, and camp.
    """
    if bit_name not in morgan_names_filt:
        return pd.DataFrame()

    filt_idx = morgan_names_filt.index(bit_name)
    orig_bit = orig_bit_indices[filt_idx]   # original 0-2047 bit index

    rows = []
    for mol_idx, binfo in enumerate(all_bit_info):
        if orig_bit in binfo:
            row = df.iloc[mol_idx]
            rows.append({
                'Molecule Name': row.get('Molecule Name', f'Mol_{mol_idx}'),
                'SMILES':        row['SMILES'],
                'gu':            row['gu'],
                'camp':          row['camp'],
            })

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values('gu', ascending=False).reset_index(drop=True)
    return result


# Print molecule lists for every top-25 consensus bit
print('\n' + '='*65)
print('MOLECULES CONTAINING EACH TOP CONSENSUS MORGAN BIT')
print('='*65)

for _, row in consensus_df.iterrows():
    bit_name  = row['Feature']
    score     = row['ConsensusScore']
    direction = '▲ HIGH GU & LOW cAMP' if score > 0 else '▼ HINDERS desired outcome'
    direction_color = '+' if score > 0 else '-'

    mol_df = molecules_with_bit(
        bit_name, names_filtered, orig_bit_indices, all_bit_info, df
    )

    print(f'\n[{direction_color}] {bit_name}  |  Bias Score: {score:+.4f}  |  {direction}')
    print(f'    Molecules with this bit active: {len(mol_df)}')

    if mol_df.empty:
        print('    (no molecules found)')
    else:
        for i, mol_row in mol_df.iterrows():
            print(f'    {i+1}. {mol_row["Molecule Name"]}  '
                  f'(GU={mol_row["gu"]:.3f}, cAMP={mol_row["camp"]:.3f})')

## 8. Morgan Bit Interpretation & SHAP Molecular Highlights

Visualise the chemical substructures behind the top consensus-scoring Morgan bits,
then overlay per-molecule SHAP values as atom-level colours.

In [ ]:
# ── (a) BitInfo structural interpretation of top consensus bits ─────────────
top_positive = consensus_df[consensus_df['ConsensusScore'] > 0]['Feature'].tolist()
top_negative = consensus_df[consensus_df['ConsensusScore'] < 0]['Feature'].tolist()

print('=== Top POSITIVE bias bits (High GU & Low cAMP) — substructure view ===')
show_top_morgan_bits(
    top_positive, df['SMILES'].tolist(), all_bit_info,
    orig_bit_indices, names_filtered, n_mols=2, n_bits=5
)

print('\n=== Top NEGATIVE bias bits — substructure view ===')
show_top_morgan_bits(
    top_negative, df['SMILES'].tolist(), all_bit_info,
    orig_bit_indices, names_filtered, n_mols=2, n_bits=5
)

# ── (b) SHAP molecular highlighting for best SVR fold ───────────────────────
# Retrieve the val-set SMILES and bitInfo for the best SVR fold
_, _, _, val_idx_svr_best = svr_fold_models[best_fold_svr]
val_smiles_svr   = [df['SMILES'].iloc[i] for i in val_idx_svr_best]
val_bitinfo_svr  = [all_bit_info[i]       for i in val_idx_svr_best]

print('\n=== SHAP Highlighting — SVR GU predictions ===')
show_shap_molecules(
    np.array(shap_svr_gu), 'SVR — GU',
    val_smiles_svr, val_bitinfo_svr,
    orig_bit_indices, names_filtered, n=4
)

print('\n=== SHAP Highlighting — SVR cAMP predictions ===')
show_shap_molecules(
    np.array(shap_svr_camp), 'SVR — cAMP',
    val_smiles_svr, val_bitinfo_svr,
    orig_bit_indices, names_filtered, n=4
)

# ── (c) SHAP highlighting for RF (TreeSHAP, larger sample available) ────────
val_smiles_rf  = [df['SMILES'].iloc[i] for i in val_idx_rf]
val_bitinfo_rf = [all_bit_info[i]       for i in val_idx_rf]

print('\n=== SHAP Highlighting — Random Forest GU predictions ===')
show_shap_molecules(
    shap_rf_gu, 'RF — GU',
    val_smiles_rf, val_bitinfo_rf,
    orig_bit_indices, names_filtered, n=4
)

### 8.1 Morgan Bit Interpretation Helper

Morgan fingerprint bits are hashed circular substructures and do not have fixed chemical definitions (unlike MACCS keys). The recommended workflow is:

1. **Identify the top-scoring bit indices** from the consensus bias DataFrame.
2. **Use `AllChem.GetMorganBitInfo()`** on a representative molecule to see which atom/radius generated each bit.
3. **Visualise fragments** with `Draw.DrawMorganBit()` for a pictorial interpretation.

The cell below implements steps 1–3 automatically for the top positive-bias bits.

In [ ]:
# ── Morgan bit fragment visualisation ──────────────────────────────────────
# For each top-bias bit, find which molecule best exemplifies it and draw
# the circular substructure it encodes.

TOP_N_BITS = 10

# Extract positive-bias top bits from consensus_df
pos_bits = consensus_df[consensus_df['ConsensusScore'] > 0].head(TOP_N_BITS)
top_bit_indices = [int(f.split('_')[1]) for f in pos_bits['Feature']]

print('Top positive-bias Morgan bits and their bit indices:')
for feat, score in zip(pos_bits['Feature'], pos_bits['ConsensusScore']):
    print(f'  {feat}  →  Consensus bias score = {score:.4f}')

# For each bit, find a molecule that has it set and render the fragment
mols_for_bits = []
legends = []

for bit_idx in top_bit_indices:
    # Search dataset for a molecule that has this bit set
    for smi in df['SMILES']:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        bi = {}  # bit_info dict
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=MORGAN_RADIUS, nBits=MORGAN_BITS, bitInfo=bi
        )
        if bit_idx in bi:
            mols_for_bits.append((mol, bit_idx, bi))
            legends.append(f'Bit {bit_idx}')
            break

if mols_for_bits:
    imgs = []
    for mol, bit_idx, bi in mols_for_bits:
        img = Draw.DrawMorganBit(mol, bit_idx, bi, molSize=(250, 250))
        imgs.append(img)

    # Display as a grid
    from PIL import Image
    import math
    ncols = min(5, len(imgs))
    nrows = math.ceil(len(imgs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()
    for ax, img, leg in zip(axes, imgs, legends):
        ax.imshow(img)
        ax.set_title(leg, fontsize=10)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')
    plt.suptitle('Top Positive-Bias Morgan Fragments', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No matching molecules found for the top bits in this dataset.')

consensus_df['BitIndex'] = consensus_df['Feature'].apply(lambda x: int(x.split('_')[1]))
print('\nFull top-feature table with bit indices:')
display(consensus_df[['Feature', 'BitIndex', 'ConsensusScore', 'SVR_bias', 'RF_bias', 'XGB_bias']].head(TOP_N_BITS))